# IUROPA observed citation edges — v3

This notebook creates a **clean judgment-to-judgment citation network** for REM work.

The retained event universe satisfies all of the following:

- the sender exists in `rem_node_features.csv`;
- the receiver exists in `rem_node_features.csv`;
- both are therefore judgments;
- `sender != receiver`;
- the cited judgment is in the **strict past** of the citing judgment:

\[
t_{receiver} < t_{sender}.
\]

This means self-citations, same-day citations, future-dated citations, and citation endpoints outside the judgment node universe are excluded from the modelling event set.

Outputs:

1. `rem_events.csv` — minimal event log for R/amorem: `event_id`, `sender`, `receiver`, `time`.
2. `rem_edge_features.csv` — the same observed citations plus **constructed dyadic features only**. It does **not** duplicate all sender and receiver node attributes.
3. `rem_edge_cleaning_summary.csv` — counts documenting the filtering.
4. `rem_edge_exclusions.csv` — excluded raw citation pairs and their reason(s).
5. `rem_edge_feature_dictionary.csv` — compact feature descriptions.

No Parquet and no large Excel workbook are produced.


**v3 fix:** avoids duplicating `decision_date` during sender/receiver node merges, which previously created suffixed date columns and caused a `KeyError` in the timing-feature step.

In [1]:
# ============================================================
# 0. CONFIGURATION
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

NODES_CSV = Path("rem_node_features.csv")
CITATIONS_CSV = Path("citations.csv")

OUTPUT_EVENTS = Path("rem_events.csv")
OUTPUT_EDGE_FEATURES = Path("rem_edge_features.csv")
OUTPUT_SUMMARY = Path("rem_edge_cleaning_summary.csv")
OUTPUT_EXCLUSIONS = Path("rem_edge_exclusions.csv")
OUTPUT_DICTIONARY = Path("rem_edge_feature_dictionary.csv")

for p in [NODES_CSV, CITATIONS_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run iuropa_data_nodes_v3.ipynb first and "
            "put citations.csv in the same folder."
        )


In [2]:
# ============================================================
# 1. LOAD COMPACT JUDGMENT NODES + RAW CITATIONS
# ============================================================

nodes = pd.read_csv(NODES_CSV)
citations_raw = pd.read_csv(CITATIONS_CSV)

required_node_cols = {"decision_id", "decision_date"}
required_edge_cols = {"citing_decision_id", "cited_decision_id"}

if missing := (required_node_cols - set(nodes.columns)):
    raise KeyError(f"rem_node_features.csv missing columns: {sorted(missing)}")

if missing := (required_edge_cols - set(citations_raw.columns)):
    raise KeyError(f"citations.csv missing columns: {sorted(missing)}")

nodes["decision_date"] = pd.to_datetime(nodes["decision_date"], errors="coerce")
nodes = nodes.drop_duplicates("decision_id").copy()

citations_raw = citations_raw.drop_duplicates(
    ["citing_decision_id", "cited_decision_id"]
).copy()

print(f"Judgment nodes:       {len(nodes):,}")
print(f"Raw citation pairs:   {len(citations_raw):,}")


Judgment nodes:       25,778
Raw citation pairs:   283,994


In [3]:
# ============================================================
# 2. HELPERS
# ============================================================

def to_set(x):
    if pd.isna(x):
        return set()
    if isinstance(x, (set, list, tuple)):
        vals = x
    else:
        vals = str(x).split(";")
    return {
        str(v).strip()
        for v in vals
        if str(v).strip()
        and str(v).strip().lower() not in {"nan", "none", "<na>"}
    }

def shared_count(a, b):
    return len(to_set(a) & to_set(b))

def jaccard(a, b):
    A, B = to_set(a), to_set(b)
    union = A | B
    if not union:
        return np.nan
    return len(A & B) / len(union)

def any_overlap(a, b):
    A, B = to_set(a), to_set(b)
    if not A and not B:
        return np.nan
    return int(bool(A & B))

def same_scalar(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return int(str(a).strip() == str(b).strip())

def abs_diff_series(a, b):
    aa = pd.to_numeric(a, errors="coerce")
    bb = pd.to_numeric(b, errors="coerce")
    return (aa - bb).abs()

def add_set_similarity(df, citing_col, cited_col, prefix):
    if citing_col not in df.columns or cited_col not in df.columns:
        return
    df[f"n_shared_{prefix}"] = [
        shared_count(a, b) for a, b in zip(df[citing_col], df[cited_col])
    ]
    df[f"same_{prefix}"] = [
        any_overlap(a, b) for a, b in zip(df[citing_col], df[cited_col])
    ]
    df[f"{prefix}_jaccard"] = [
        jaccard(a, b) for a, b in zip(df[citing_col], df[cited_col])
    ]


In [4]:
# ============================================================
# 3. CLASSIFY AND FILTER RAW CITATIONS
# ============================================================
#
# Membership in rem_node_features.csv means membership in the retained
# judgment-only node universe.

node_ids = set(nodes["decision_id"])

check = citations_raw.copy()
check["sender_is_judgment_node"] = check["citing_decision_id"].isin(node_ids)
check["receiver_is_judgment_node"] = check["cited_decision_id"].isin(node_ids)

date_lookup = nodes.set_index("decision_id")["decision_date"]

check["citing_decision_date"] = check["citing_decision_id"].map(date_lookup)
check["cited_decision_date"] = check["cited_decision_id"].map(date_lookup)

check["is_self_citation"] = (
    check["citing_decision_id"] == check["cited_decision_id"]
)

check["is_same_day"] = (
    check["citing_decision_date"].notna()
    & check["cited_decision_date"].notna()
    & (check["citing_decision_date"] == check["cited_decision_date"])
)

check["is_future_citation"] = (
    check["citing_decision_date"].notna()
    & check["cited_decision_date"].notna()
    & (check["cited_decision_date"] > check["citing_decision_date"])
)

check["is_strict_past"] = (
    check["citing_decision_date"].notna()
    & check["cited_decision_date"].notna()
    & (check["cited_decision_date"] < check["citing_decision_date"])
)

# Retained modelling event:
# judgment -> judgment, not self, and receiver strictly predates sender.
check["keep_rem_event"] = (
    check["sender_is_judgment_node"]
    & check["receiver_is_judgment_node"]
    & ~check["is_self_citation"]
    & check["is_strict_past"]
)

def exclusion_reason(r):
    reasons = []
    if not r["sender_is_judgment_node"]:
        reasons.append("sender_not_in_judgment_nodes")
    if not r["receiver_is_judgment_node"]:
        reasons.append("receiver_not_in_judgment_nodes")
    if r["is_self_citation"]:
        reasons.append("self_citation")
    if r["is_same_day"] and not r["is_self_citation"]:
        reasons.append("same_day_distinct_judgments")
    if r["is_future_citation"]:
        reasons.append("future_dated_citation")
    if (
        r["sender_is_judgment_node"]
        and r["receiver_is_judgment_node"]
        and pd.isna(r["citing_decision_date"])
    ):
        reasons.append("missing_sender_date")
    if (
        r["sender_is_judgment_node"]
        and r["receiver_is_judgment_node"]
        and pd.isna(r["cited_decision_date"])
    ):
        reasons.append("missing_receiver_date")
    return ";".join(reasons) if reasons else ""

check["exclusion_reason"] = check.apply(exclusion_reason, axis=1)

valid_pairs = check.loc[check["keep_rem_event"]].copy()
excluded = check.loc[~check["keep_rem_event"]].copy()

print(f"Raw citation pairs:             {len(check):,}")
print(f"Retained judgment events:       {len(valid_pairs):,}")
print(f"Excluded citation pairs:        {len(excluded):,}")
print()
print("Exclusion flags:")
for c in [
    "sender_is_judgment_node",
    "receiver_is_judgment_node",
    "is_self_citation",
    "is_same_day",
    "is_future_citation",
]:
    print(f"  {c:32s} {int(check[c].sum()):,}")


Raw citation pairs:             283,994
Retained judgment events:       260,252
Excluded citation pairs:        23,742

Exclusion flags:
  sender_is_judgment_node          267,095
  receiver_is_judgment_node        283,040
  is_self_citation                 2,805
  is_same_day                      5,040
  is_future_citation               925


In [5]:
# ============================================================
# 4. BUILD CLEAN EVENT LOG
# ============================================================

valid_pairs = valid_pairs.sort_values(
    ["citing_decision_date", "citing_decision_id", "cited_decision_id"]
).reset_index(drop=True)

valid_pairs["event_id"] = [
    f"E{i:09d}" for i in range(1, len(valid_pairs) + 1)
]

rem_events = valid_pairs.rename(columns={
    "citing_decision_id": "sender",
    "cited_decision_id": "receiver",
    "citing_decision_date": "time",
})[
    ["event_id", "sender", "receiver", "time"]
].copy()

if rem_events.duplicated(["sender", "receiver"]).any():
    raise ValueError("Duplicate sender-receiver pairs remain in rem_events.")

print(f"Clean REM event log: {len(rem_events):,} events")
display(rem_events.head())


Clean REM event log: 260,252 events


,event_id,sender,receiver,time
0,E000000001,DOC_JUDG_CJ_1955_0005_19550628,DOC_JUDG_CJ_1954_0001_19541221,1955-06-28
1,E000000002,DOC_JUDG_CJ_1955_0005_19550628,DOC_JUDG_CJ_1954_0002_19541221,1955-06-28
2,E000000003,DOC_JUDG_CJ_1955_0005_19550628,DOC_JUDG_CJ_1954_0003_19550211,1955-06-28
3,E000000004,DOC_JUDG_CJ_1954_0007_19560423,DOC_JUDG_CJ_1954_0001_19541221,1956-04-23
4,E000000005,DOC_JUDG_CJ_1954_0008_19560423,DOC_JUDG_CJ_1954_0001_19541221,1956-04-23


In [6]:
# ============================================================
# 5. ATTACH ONLY THE NODE FIELDS NEEDED TO CONSTRUCT DYAD FEATURES
# ============================================================

# IMPORTANT:
# decision_date is deliberately NOT included here because the clean event table
# already carries citing_decision_date and cited_decision_date. Including it again
# during the sender/receiver merges would create *_x / *_y columns and break the
# timing-feature cell below.
needed_node_candidates = [
    "decision_id", "court",
    "case_id", "proceeding_id",
    "procedures", "outcomes",
    "is_preliminary_ruling", "is_appeal", "is_annulment",
    "is_infringement", "is_damages_action", "is_failure_to_act",
    "joined_proceeding_list", "appeal_of_list",
    "chambers", "is_grand_chamber", "is_full_court_or_plenary",
    "panel_judge_ids", "panel_size", "panel_member_states",
    "panel_mean_age_approx", "panel_share_female",
    "panel_member_state_diversity",
    "panel_mean_cjeu_service_years",
    "panel_mean_prior_judge_assignments",
    "rapporteur_ids", "rapporteur_age_approx",
    "rapporteur_cjeu_service_years",
    "rapporteur_prior_judge_assignments",
    "ag_ids", "ag_age_approx", "ag_cjeu_service_years",
    "ag_prior_assignments",
    "directory_codes", "directory_sections",
    "directory_major_labels", "directory_minor_labels",
    "keywords",
    "directory_code_novelty", "directory_section_novelty",
    "directory_major_novelty", "keyword_novelty",
    "party_ids", "party_types",
    "observer_ids", "observer_types",
    "has_commission_party", "had_observers",
    "referring_court_ids", "referring_member_states",
    "referring_court_levels", "referring_court_prestiges",
    "referring_court_collegialitys",
    "referring_court_competences",
    "referring_court_competence_detaileds",
]

needed_node_cols = [c for c in needed_node_candidates if c in nodes.columns]
node_subset = nodes[needed_node_cols].copy()

sender_nodes = node_subset.rename(
    columns={
        c: f"citing_{c}"
        for c in node_subset.columns
        if c != "decision_id"
    }
).rename(columns={"decision_id": "citing_decision_id"})

receiver_nodes = node_subset.rename(
    columns={
        c: f"cited_{c}"
        for c in node_subset.columns
        if c != "decision_id"
    }
).rename(columns={"decision_id": "cited_decision_id"})

work = valid_pairs[
    ["event_id", "citing_decision_id", "cited_decision_id",
     "citing_decision_date", "cited_decision_date"]
].copy()

work = work.merge(
    sender_nodes,
    on="citing_decision_id",
    how="left",
    validate="many_to_one",
)

work = work.merge(
    receiver_nodes,
    on="cited_decision_id",
    how="left",
    validate="many_to_one",
)

print(f"Feature-construction working table: {work.shape[0]:,} rows")

# Sanity check: dates must exist exactly once under the expected names.
required_date_cols = {"citing_decision_date", "cited_decision_date"}
missing_date_cols = required_date_cols - set(work.columns)
if missing_date_cols:
    raise KeyError(
        f"Expected date columns missing after node merges: {sorted(missing_date_cols)}"
    )

date_like_cols = [c for c in work.columns if "decision_date" in c]
print("Decision-date columns after merge:", date_like_cols)

unexpected_date_cols = [
    c for c in date_like_cols
    if c not in {"citing_decision_date", "cited_decision_date"}
]
if unexpected_date_cols:
    raise ValueError(
        "Unexpected duplicate date columns created during merge: "
        f"{unexpected_date_cols}"
    )


Feature-construction working table: 260,252 rows
Decision-date columns after merge: ['citing_decision_date', 'cited_decision_date']


In [7]:
# ============================================================
# 6. TIMING FEATURES
# ============================================================

work["citation_lag_days"] = (
    pd.to_datetime(work["citing_decision_date"])
    - pd.to_datetime(work["cited_decision_date"])
).dt.days
work["citation_lag_years"] = work["citation_lag_days"] / 365.25


In [8]:
# ============================================================
# 7. COURT / FORMATION / PERSONNEL DYAD FEATURES
# ============================================================

if {"citing_court", "cited_court"}.issubset(work.columns):
    work["same_court"] = [
        same_scalar(a, b)
        for a, b in zip(work["citing_court"], work["cited_court"])
    ]

if {"citing_panel_size", "cited_panel_size"}.issubset(work.columns):
    work["panel_size_abs_diff"] = abs_diff_series(
        work["citing_panel_size"], work["cited_panel_size"]
    )

for c1, c2, prefix in [
    ("citing_chambers", "cited_chambers", "chamber"),
    ("citing_panel_judge_ids", "cited_panel_judge_ids", "panel_judges"),
    ("citing_panel_member_states", "cited_panel_member_states", "panel_member_states"),
    ("citing_rapporteur_ids", "cited_rapporteur_ids", "rapporteur"),
    ("citing_ag_ids", "cited_ag_ids", "advocate_general"),
]:
    add_set_similarity(work, c1, c2, prefix)

if {"citing_rapporteur_ids", "cited_panel_judge_ids"}.issubset(work.columns):
    work["citing_rapporteur_sat_on_cited_panel"] = [
        any_overlap(a, b)
        for a, b in zip(work["citing_rapporteur_ids"], work["cited_panel_judge_ids"])
    ]

if {"cited_rapporteur_ids", "citing_panel_judge_ids"}.issubset(work.columns):
    work["cited_rapporteur_sits_on_citing_panel"] = [
        any_overlap(a, b)
        for a, b in zip(work["cited_rapporteur_ids"], work["citing_panel_judge_ids"])
    ]

for base in [
    "panel_mean_age_approx",
    "panel_share_female",
    "panel_member_state_diversity",
    "panel_mean_cjeu_service_years",
    "panel_mean_prior_judge_assignments",
    "rapporteur_age_approx",
    "rapporteur_cjeu_service_years",
    "rapporteur_prior_judge_assignments",
    "ag_age_approx",
    "ag_cjeu_service_years",
    "ag_prior_assignments",
]:
    c1, c2 = f"citing_{base}", f"cited_{base}"
    if c1 in work.columns and c2 in work.columns:
        work[f"{base}_abs_diff"] = abs_diff_series(work[c1], work[c2])


In [9]:
# ============================================================
# 8. LEGAL-SUBJECT / PROCEDURE DYAD FEATURES
# ============================================================

for c1, c2, prefix in [
    ("citing_procedures", "cited_procedures", "procedures"),
    ("citing_outcomes", "cited_outcomes", "outcomes"),
    ("citing_directory_codes", "cited_directory_codes", "directory_codes"),
    ("citing_directory_sections", "cited_directory_sections", "directory_sections"),
    ("citing_directory_major_labels", "cited_directory_major_labels", "directory_major_labels"),
    ("citing_directory_minor_labels", "cited_directory_minor_labels", "directory_minor_labels"),
    ("citing_keywords", "cited_keywords", "keywords"),
]:
    add_set_similarity(work, c1, c2, prefix)

for flag in [
    "is_preliminary_ruling",
    "is_appeal",
    "is_annulment",
    "is_infringement",
    "is_damages_action",
    "is_failure_to_act",
]:
    c1, c2 = f"citing_{flag}", f"cited_{flag}"
    if c1 in work.columns and c2 in work.columns:
        a = pd.to_numeric(work[c1], errors="coerce")
        b = pd.to_numeric(work[c2], errors="coerce")
        work[f"both_{flag.replace('is_', '')}"] = np.where(
            a.notna() & b.notna(),
            ((a == 1) & (b == 1)).astype(int),
            np.nan
        )

for flag in ["is_grand_chamber", "is_full_court_or_plenary"]:
    c1, c2 = f"citing_{flag}", f"cited_{flag}"
    if c1 in work.columns and c2 in work.columns:
        a = pd.to_numeric(work[c1], errors="coerce")
        b = pd.to_numeric(work[c2], errors="coerce")
        work[f"both_{flag.replace('is_', '')}"] = np.where(
            a.notna() & b.notna(),
            ((a == 1) & (b == 1)).astype(int),
            np.nan
        )


In [10]:
# ============================================================
# 9. PARTIES / OBSERVERS / NATIONAL COURTS
# ============================================================

for c1, c2, prefix in [
    ("citing_party_ids", "cited_party_ids", "parties"),
    ("citing_party_types", "cited_party_types", "party_types"),
    ("citing_observer_ids", "cited_observer_ids", "observers"),
    ("citing_observer_types", "cited_observer_types", "observer_types"),
    ("citing_referring_court_ids", "cited_referring_court_ids", "referring_courts"),
    ("citing_referring_member_states", "cited_referring_member_states", "referring_member_states"),
    ("citing_referring_court_levels", "cited_referring_court_levels", "referring_court_levels"),
    ("citing_referring_court_prestiges", "cited_referring_court_prestiges", "referring_court_prestiges"),
    ("citing_referring_court_collegialitys", "cited_referring_court_collegialitys", "referring_court_collegialitys"),
    ("citing_referring_court_competences", "cited_referring_court_competences", "referring_court_competences"),
    ("citing_referring_court_competence_detaileds",
     "cited_referring_court_competence_detaileds",
     "referring_court_competence_detaileds"),
]:
    add_set_similarity(work, c1, c2, prefix)

if "same_referring_courts" in work.columns:
    work["same_referring_national_court"] = work["same_referring_courts"]

if "same_referring_member_states" in work.columns:
    work["same_referring_country"] = work["same_referring_member_states"]

if {"citing_has_commission_party", "cited_has_commission_party"}.issubset(work.columns):
    a = pd.to_numeric(work["citing_has_commission_party"], errors="coerce")
    b = pd.to_numeric(work["cited_has_commission_party"], errors="coerce")
    work["commission_party_in_both"] = np.where(
        a.notna() & b.notna(), ((a == 1) & (b == 1)).astype(int), np.nan
    )

if {"citing_had_observers", "cited_had_observers"}.issubset(work.columns):
    a = pd.to_numeric(work["citing_had_observers"], errors="coerce")
    b = pd.to_numeric(work["cited_had_observers"], errors="coerce")
    work["both_had_observers"] = np.where(
        a.notna() & b.notna(), ((a == 1) & (b == 1)).astype(int), np.nan
    )


C:\Users\laze\AppData\Local\Temp\ipykernel_25668\3275380638.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{prefix}_jaccard"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_25668\3639752626.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work["same_referring_national_court"] = work["same_referring_courts"]
C:\Users\laze\AppData\Local\Temp\ipykernel_25668\3639752626.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [11]:
# ============================================================
# 10. PROCEDURAL-LINEAGE DYAD FEATURES
# ============================================================

if {"citing_case_id", "cited_case_id"}.issubset(work.columns):
    work["same_case"] = [
        same_scalar(a, b)
        for a, b in zip(work["citing_case_id"], work["cited_case_id"])
    ]

if {"citing_proceeding_id", "cited_proceeding_id"}.issubset(work.columns):
    work["same_proceeding"] = [
        same_scalar(a, b)
        for a, b in zip(work["citing_proceeding_id"], work["cited_proceeding_id"])
    ]

if {"citing_appeal_of_list", "cited_proceeding_id"}.issubset(work.columns):
    work["citing_is_appeal_of_cited_proceeding"] = [
        int(str(proc).strip() in to_set(lst)) if pd.notna(proc) else np.nan
        for lst, proc in zip(work["citing_appeal_of_list"], work["cited_proceeding_id"])
    ]

if {"citing_proceeding_id", "cited_proceeding_id"}.issubset(work.columns):
    citing_joined = work.get(
        "citing_joined_proceeding_list",
        pd.Series(np.nan, index=work.index)
    )
    cited_joined = work.get(
        "cited_joined_proceeding_list",
        pd.Series(np.nan, index=work.index)
    )

    citing_bundle = [
        to_set(joined) | ({str(proc).strip()} if pd.notna(proc) else set())
        for joined, proc in zip(citing_joined, work["citing_proceeding_id"])
    ]
    cited_bundle = [
        to_set(joined) | ({str(proc).strip()} if pd.notna(proc) else set())
        for joined, proc in zip(cited_joined, work["cited_proceeding_id"])
    ]

    work["same_joined_proceeding_family"] = [
        int(bool(a & b)) for a, b in zip(citing_bundle, cited_bundle)
    ]
    work["joined_proceeding_family_jaccard"] = [
        len(a & b) / len(a | b) if (a | b) else np.nan
        for a, b in zip(citing_bundle, cited_bundle)
    ]


C:\Users\laze\AppData\Local\Temp\ipykernel_25668\178970126.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work["same_case"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_25668\178970126.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work["same_proceeding"] = [
C:\Users\laze\AppData\Local\Temp\ipykernel_25668\178970126.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [12]:
# ============================================================
# 11. NOVELTY-DIFFERENCE FEATURES
# ============================================================

for base in [
    "directory_code_novelty",
    "directory_section_novelty",
    "directory_major_novelty",
    "keyword_novelty",
]:
    c1, c2 = f"citing_{base}", f"cited_{base}"
    if c1 in work.columns and c2 in work.columns:
        work[f"{base}_abs_diff"] = abs_diff_series(work[c1], work[c2])


C:\Users\laze\AppData\Local\Temp\ipykernel_25668\2399997015.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work[f"{base}_abs_diff"] = abs_diff_series(work[c1], work[c2])
C:\Users\laze\AppData\Local\Temp\ipykernel_25668\2399997015.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work[f"{base}_abs_diff"] = abs_diff_series(work[c1], work[c2])
C:\Users\laze\AppData\Local\Temp\ipykernel_25668\2399997015.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

In [13]:
# ============================================================
# 12. KEEP ONLY IDs / DATES + ACTUALLY CONSTRUCTED EDGE FEATURES
# ============================================================
#
# Crucially, no copied citing_* or cited_* node attributes are exported.

base_cols = [
    "event_id",
    "citing_decision_id",
    "cited_decision_id",
    "citing_decision_date",
    "cited_decision_date",
    "citation_lag_days",
    "citation_lag_years",
]

constructed_cols = [
    c for c in work.columns
    if c not in base_cols
    and not c.startswith("citing_")
    and not c.startswith("cited_")
]

rem_edge_features = work[base_cols + constructed_cols].copy()

rem_edge_features = rem_edge_features.rename(columns={
    "citing_decision_id": "sender",
    "cited_decision_id": "receiver",
    "citing_decision_date": "time",
    "cited_decision_date": "receiver_date",
})

print(
    f"Compact observed edge-feature table: "
    f"{len(rem_edge_features):,} rows x {rem_edge_features.shape[1]:,} columns"
)
display(rem_edge_features.head())


Compact observed edge-feature table: 260,252 rows x 109 columns


,event_id,sender,receiver,time,receiver_date,citation_lag_days,citation_lag_years,same_court,panel_size_abs_diff,n_shared_chamber,...,commission_party_in_both,both_had_observers,same_case,same_proceeding,same_joined_proceeding_family,joined_proceeding_family_jaccard,directory_code_novelty_abs_diff,directory_section_novelty_abs_diff,directory_major_novelty_abs_diff,keyword_novelty_abs_diff
0,E000000001,DOC_JUDG_CJ_1955_0005_19550628,DOC_JUDG_CJ_1954_0001_19541221,1955-06-28,1954-12-21,189,0.517454,1,0.0,1,...,0.0,0.0,0,0,0,0.0,0.000000,1.0,1.000000,1.0
1,E000000002,DOC_JUDG_CJ_1955_0005_19550628,DOC_JUDG_CJ_1954_0002_19541221,1955-06-28,1954-12-21,189,0.517454,1,0.0,1,...,0.0,0.0,0,0,0,0.0,0.000000,1.0,1.000000,1.0
2,E000000003,DOC_JUDG_CJ_1955_0005_19550628,DOC_JUDG_CJ_1954_0003_19550211,1955-06-28,1955-02-11,137,0.375086,1,0.0,1,...,0.0,0.0,0,0,0,0.0,0.500000,0.0,0.500000,0.0
3,E000000004,DOC_JUDG_CJ_1954_0007_19560423,DOC_JUDG_CJ_1954_0001_19541221,1956-04-23,1954-12-21,489,1.338809,1,0.0,1,...,NaN,0.0,0,0,0,0.0,0.428571,1.0,0.750000,1.0
4,E000000005,DOC_JUDG_CJ_1954_0008_19560423,DOC_JUDG_CJ_1954_0001_19541221,1956-04-23,1954-12-21,489,1.338809,1,0.0,1,...,NaN,0.0,0,0,0,0.0,0.166667,1.0,0.666667,1.0


In [14]:
# ============================================================
# 13. CLEANING SUMMARY + FEATURE DICTIONARY
# ============================================================

summary_rows = [
    ("raw_unique_citation_pairs", len(check)),
    ("sender_in_judgment_nodes", int(check["sender_is_judgment_node"].sum())),
    ("receiver_in_judgment_nodes", int(check["receiver_is_judgment_node"].sum())),
    ("self_citations", int(check["is_self_citation"].sum())),
    ("same_day_citations_including_self", int(check["is_same_day"].sum())),
    ("future_dated_citations", int(check["is_future_citation"].sum())),
    ("retained_strict_past_judgment_events", int(check["keep_rem_event"].sum())),
    ("excluded_total", int((~check["keep_rem_event"]).sum())),
]

rem_edge_cleaning_summary = pd.DataFrame(
    summary_rows, columns=["metric", "n"]
)
rem_edge_cleaning_summary["pct_of_raw"] = (
    100 * rem_edge_cleaning_summary["n"] / len(check)
)

feature_dictionary = pd.DataFrame({
    "feature": rem_edge_features.columns,
    "description": [
        {
            "event_id": "Stable row identifier for the retained observed citation.",
            "sender": "Citing judgment decision_id.",
            "receiver": "Cited judgment decision_id.",
            "time": "Date of the citing judgment / citation event time.",
            "receiver_date": "Date the cited judgment entered the network.",
            "citation_lag_days": "Days between receiver and sender judgments.",
            "citation_lag_years": "citation_lag_days / 365.25.",
        }.get(
            c,
            "Constructed dyadic citation feature; see the corresponding notebook section."
        )
        for c in rem_edge_features.columns
    ],
})

display(rem_edge_cleaning_summary)


,metric,n,pct_of_raw
0,raw_unique_citation_pairs,283994,100.000000
1,sender_in_judgment_nodes,267095,94.049522
2,receiver_in_judgment_nodes,283040,99.664077
3,self_citations,2805,0.987697
4,same_day_citations_including_self,5040,1.774685
5,future_dated_citations,925,0.325711
6,retained_strict_past_judgment_events,260252,91.639964
7,excluded_total,23742,8.360036


In [15]:
# ============================================================
# 14. EXPORT SMALL CSV OUTPUTS
# ============================================================

# Format dates as ISO strings for portable R/Python import.
for df in [rem_events, rem_edge_features, excluded]:
    for c in ["time", "receiver_date", "citing_decision_date", "cited_decision_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.strftime("%Y-%m-%d")

rem_events.to_csv(
    OUTPUT_EVENTS,
    index=False,
    encoding="utf-8"
)

rem_edge_features.to_csv(
    OUTPUT_EDGE_FEATURES,
    index=False,
    encoding="utf-8"
)

rem_edge_cleaning_summary.to_csv(
    OUTPUT_SUMMARY,
    index=False,
    encoding="utf-8"
)

excluded.to_csv(
    OUTPUT_EXCLUSIONS,
    index=False,
    encoding="utf-8"
)

feature_dictionary.to_csv(
    OUTPUT_DICTIONARY,
    index=False,
    encoding="utf-8"
)

print("Saved:")
print(f"  {OUTPUT_EVENTS.resolve()}")
print(f"  {OUTPUT_EDGE_FEATURES.resolve()}")
print(f"  {OUTPUT_SUMMARY.resolve()}")
print(f"  {OUTPUT_EXCLUSIONS.resolve()}")
print(f"  {OUTPUT_DICTIONARY.resolve()}")
print()
print(f"rem_events.csv:        {len(rem_events):,} rows x {rem_events.shape[1]:,} columns")
print(f"rem_edge_features.csv: {len(rem_edge_features):,} rows x {rem_edge_features.shape[1]:,} columns")


Saved:
  C:\Users\laze\Downloads\IUROPA\IUROPA\rem_events.csv
  C:\Users\laze\Downloads\IUROPA\IUROPA\rem_edge_features.csv
  C:\Users\laze\Downloads\IUROPA\IUROPA\rem_edge_cleaning_summary.csv
  C:\Users\laze\Downloads\IUROPA\IUROPA\rem_edge_exclusions.csv
  C:\Users\laze\Downloads\IUROPA\IUROPA\rem_edge_feature_dictionary.csv

rem_events.csv:        260,252 rows x 4 columns
rem_edge_features.csv: 260,252 rows x 109 columns
